In [ ]:
import mne
import os
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- 1. 기본 설정 ---
DATA_PATH = "../Dataset/EEG/" # 사용자 설정 경로
TASK_NAME = 'task-MIvsRest'
RUN_ID = 'run-1'

# 이벤트 ID와 설명 매핑 (레이블 변경)
event_name_labels = {
    'Rest': 'OVTK_GDF_Tongue',           # 'Rest'에 해당하는 원래 어노테이션
    'Motor Imagery': 'OVTK_GDF_Right'   # 'Motor Imagery'에 해당하는 원래 어노테이션
}
event_id_map_edf = { # EDF 어노테이션 문자열을 내부 ID로 매핑 (mne.events_from_annotations용)
    'OVTK_GDF_Tongue': 9, # Rest
    'OVTK_GDF_Right': 7   # Motor Imagery
}
# 내부 ID를 새로운 레이블로 매핑 (최종 결과물 및 플롯용)
id_to_new_label_map = {
    9: 'Rest',
    7: 'Motor Imagery'
}

# --- 2. 모든 피험자 ID 가져오기 ---
participants_tsv_path = os.path.join(DATA_PATH, 'participants.tsv')
try:
    participants_info = pd.read_csv(participants_tsv_path, sep='\t')
    subject_ids = participants_info['participant_id'].tolist()
    # 사용자가 제공한 실제 피험자 리스트로 필터링 (선택 사항)
    # subject_ids = [sid for sid in subject_ids if sid in ['sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-12']]
    print(f"총 {len(subject_ids)}명의 피험자를 찾았습니다: {subject_ids}")
except FileNotFoundError:
    print(f"경고: participants.tsv 파일을 찾을 수 없습니다. 경로: {participants_tsv_path}")
    subject_ids = ['sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-12']
    print(f"지정된 피험자 리스트 {subject_ids}를 사용합니다.")
except Exception as e:
    print(f"participants.tsv 파일 로드 중 오류: {e}")
    subject_ids = ['sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-12']
    print(f"지정된 피험자 리스트 {subject_ids}를 사용합니다.")

In [ ]:
# --- 3. 각 피험자 데이터의 이벤트 상황 체크 및 all_subjects_block_details 생성 ---
all_subjects_block_details = {}

for sub_id in subject_ids:
    print(f"\n\n--- 피험자 {sub_id}, {TASK_NAME}, {RUN_ID} 이벤트 상황 체크 시작 ---")
    eeg_file_name = f"{sub_id}_{TASK_NAME}_{RUN_ID}_eeg.edf"
    eeg_file_path = os.path.join(DATA_PATH, sub_id, 'eeg', eeg_file_name)

    if not os.path.exists(eeg_file_path):
        print(f"  파일을 찾을 수 없음: {eeg_file_path}. 다음 피험자로 넘어갑니다.")
        all_subjects_block_details[sub_id] = {"error": "파일 없음"}
        continue

    try:
        raw = mne.io.read_raw_edf(eeg_file_path, preload=True, verbose='ERROR')
        events, event_dict_from_file = mne.events_from_annotations(raw, event_id=event_id_map_edf, verbose='ERROR')
        
        # event_dict_from_file의 키를 새로운 레이블로 변환 (출력용)
        # 예: {np.str_('OVTK_GDF_Right'): 7} -> {'Motor Imagery': 7}
        # 참고: event_dict_from_file의 키는 EDF 파일 내 실제 어노테이션 문자열입니다.
        #       id_to_new_label_map은 내부 ID를 최종 원하는 레이블로 바꿀 때 사용합니다.

        print(f"  추출된 총 이벤트 마커 수: {len(events)}")
        if len(events) == 0:
            print("  추출된 이벤트가 없습니다.")
            all_subjects_block_details[sub_id] = {"error": "이벤트 없음"}
            continue

        print("  이벤트 유형별 발생 횟수 (새로운 레이블 기준):")
        event_counts = pd.Series(events[:, 2]).value_counts() # events[:, 2]는 내부 ID (7 또는 9)
        for internal_event_id, count in event_counts.items():
            display_label = id_to_new_label_map.get(int(internal_event_id), f"UnknownID_{int(internal_event_id)}")
            print(f"    {display_label} (ID: {int(internal_event_id)}): {count}회")

        subject_block_info_list = []
        print("\n  각 상태 블록 세부 정보 (새로운 레이블 기준):")
        for i in range(len(events)):
            event_sample = events[i, 0]
            internal_event_id = events[i, 2] # 내부 ID (7 또는 9)
            
            # 내부 ID를 새로운 표시용 레이블로 변환
            block_display_label = id_to_new_label_map.get(int(internal_event_id), f"UnknownID_{int(internal_event_id)}")

            tmin_sec = event_sample / raw.info['sfreq']
            if i < len(events) - 1:
                tmax_sec = (events[i+1, 0] - 1) / raw.info['sfreq']
            else:
                tmax_sec = raw.times[-1]
            duration_sec = tmax_sec - tmin_sec

            if duration_sec > 0:
                block_info = {
                    'name': block_display_label, # 새로운 레이블 사용
                    'id': int(internal_event_id),
                    'start_sec': round(tmin_sec, 2),
                    'end_sec': round(tmax_sec, 2),
                    'duration_sec': round(duration_sec, 2)
                }
                subject_block_info_list.append(block_info)
                print(f"    블록 {i+1}: {block_info['name']} (ID: {block_info['id']}), "
                      f"시작: {block_info['start_sec']:.2f}s, 종료: {block_info['end_sec']:.2f}s, "
                      f"지속: {block_info['duration_sec']:.2f}s")
            # else: 경고 메시지 (이전과 동일)
        all_subjects_block_details[sub_id] = subject_block_info_list
    except Exception as e:
        print(f"  피험자 {sub_id} 처리 중 오류 발생: {e}")
        all_subjects_block_details[sub_id] = {"error": f"오류: {e}"}

print("\n--- 모든 피험자 이벤트 상황 체크 완료 (또는 시도 완료) ---")

# --- 4. 생성된 딕셔너리를 JSON 파일로 저장 ---
output_derivatives_dir = os.path.join(DATA_PATH, 'derivatives')
os.makedirs(output_derivatives_dir, exist_ok=True)
block_details_file_path = os.path.join(output_derivatives_dir, 'all_subjects_event_details.json') # 파일명 변경

try:
    with open(block_details_file_path, 'w') as f:
        json.dump(all_subjects_block_details, f, indent=4)
    print(f"\n이벤트 블록 상세 정보가 다음 경로에 저장되었습니다: {block_details_file_path}")
except Exception as e:
    print(f"\n파일 저장 중 오류 발생: {e}")

In [ ]:
# --- 5. 저장된 JSON 파일 로드 및 선택 피험자 시각화 ---
def plot_event_blocks_from_json(json_file_path, subject_id_to_plot, new_event_color_map):
    """
    저장된 JSON 파일에서 이벤트 블록 정보를 로드하여 특정 피험자에 대해 시각화합니다.
    """
    loaded_block_details = None
    if os.path.exists(json_file_path):
        try:
            with open(json_file_path, 'r') as f:
                loaded_block_details = json.load(f)
            print(f"파일에서 이벤트 블록 상세 정보를 성공적으로 불러왔습니다: {json_file_path}")
        except Exception as e:
            print(f"파일 불러오기 중 오류 발생: {e}")
            return
    else:
        print(f"저장된 이벤트 블록 상세 정보 파일을 찾을 수 없습니다: {json_file_path}")
        return

    if not loaded_block_details or subject_id_to_plot not in loaded_block_details or \
       not isinstance(loaded_block_details.get(subject_id_to_plot), list):
        print(f"피험자 {subject_id_to_plot}에 대한 유효한 블록 정보가 로드되지 않았습니다.")
        return

    blocks_to_plot = loaded_block_details[subject_id_to_plot]
    if not blocks_to_plot:
        print(f"피험자 {subject_id_to_plot}의 블록 정보가 비어있습니다.")
        return

    fig, ax = plt.subplots(figsize=(20, 2)) # 높이 조절
    max_time = 0
    default_plot_color = 'lightgrey'

    for block in blocks_to_plot:
        start_time = block['start_sec']
        end_time = block['end_sec']
        event_label_name = block['name'] # JSON에 저장된 새로운 레이블 사용
        
        color = new_event_color_map.get(event_label_name, default_plot_color)
        ax.axvspan(start_time, end_time, color=color, alpha=0.5, ymin=0, ymax=1)
        if end_time > max_time:
            max_time = end_time
    
    ax.set_ylim(0, 1)
    ax.set_yticks([]) # Y축 눈금 및 레이블 제거하여 더 깔끔하게
    ax.set_ylabel("Task", rotation=0, labelpad=20, verticalalignment='center')


    final_xlim_max = max_time if max_time > 0 else 10
    ax.set_xlim(0, final_xlim_max)
    ax.set_xlabel("Time (seconds)")
    ax.set_title(f"Event Block Timing for {subject_id_to_plot}")

    patches = []
    # 사용된 레이블과 new_event_color_map을 사용하여 범례 생성
    unique_event_labels_in_plot = sorted(list(set(b['name'] for b in blocks_to_plot if b['name'] in new_event_color_map)))
    for label in unique_event_labels_in_plot:
         patches.append(mpatches.Patch(color=new_event_color_map[label], label=label, alpha=0.5))
    
    if patches:
        ax.legend(handles=patches, loc='upper right')

    plt.grid(axis='x', linestyle=':', color='gray')
    plt.tight_layout()
    plt.show()

# --- 시각화 실행 ---
# 새로운 레이블에 대한 색상 맵
new_color_map_for_plot = {
    'Rest': 'lightcoral',
    'Motor Imagery': 'cornflowerblue'
}

# 저장된 파일 경로 (위의 저장 코드에서 사용한 경로와 동일해야 함)
event_details_json_path = os.path.join(DATA_PATH, 'derivatives', 'all_subjects_event_details.json')

plot_event_blocks_from_json(event_details_json_path, 
                            subject_id_to_plot='sub-02', # 여기서 보고 싶은 피험자 ID 지정
                            new_event_color_map=new_color_map_for_plot)